In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import string
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('wordnet')
import gensim
from gensim.utils import simple_preprocess
import spacy
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ethan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
synthetic_df = pd.read_csv('SyntheticDataset.csv', usecols = ['StudentID', 'Topic', 'StudentFeedback', 'Timestamp'])

In [4]:
def remove_punctuation(text):
    for char in text:
        if char in string.punctuation:
            text = text.replace(char, '')
    return text

In [5]:
synthetic_df['StudentFeedback'] = synthetic_df['StudentFeedback'].apply(lambda x:remove_punctuation(x))

In [6]:
stopwords_list = stopwords.words('english')
stopwords_list.extend(['ok','okay','see', 'ah', 'way', 'hi', 'hello', 'hey', 'please', 'thank', 'thanks', 'would', 'like', 'know', 'can', 'could', 'get', 'got', 'go', 'going', 'want', 'wanna', 'wont', 'will', 'wouldnt', 'should', 'shouldnt', 'im'])

In [7]:
def stopword_clean(text):
    word_list=[]
    for word in gensim.utils.simple_preprocess(text):
        if word not in stopwords_list:
            word_list.append(word)
    final_list = ' '.join(word_list)
    return final_list

In [8]:
synthetic_df['stopword_clean'] = synthetic_df['StudentFeedback'].apply(lambda x:stopword_clean(x))
synthetic_df = synthetic_df.dropna(subset=['stopword_clean'])
synthetic_df = synthetic_df.reset_index(drop=True)

In [9]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [10]:
def preprocess_text(text):
    # Tokenize
    words = word_tokenize(text.lower())  # Convert to lowercase and tokenize
    
    # Remove punctuation and non-alphabetic characters
    words = [word for word in words if word.isalpha()]
    
    # Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]
    
    # Join back into a string
    return " ".join(words)

In [11]:
synthetic_df['cleaned_text'] = synthetic_df['stopword_clean'].apply(preprocess_text)

synthetic_df = synthetic_df[synthetic_df['cleaned_text'].str.split().str.len().between(3, 50)]

synthetic_df = synthetic_df[['StudentID', 'Topic', 'Timestamp', 'cleaned_text']]

In [12]:
synthetic_df.to_csv('synthetic_cleaned.csv', index=False)